In [ ]:
# @title Mount Google Drive

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/NLP")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/NLP")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Non trovo la cartella progetto. "
        "Controlla se il path è /content/drive/MyDrive/NLP oppure modifica PROJECT_ROOT manualmente."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/NLP")

SCI_Q_LOG = PROJECT_ROOT / "logs/experiments/RUN300_science_nature.json"
WIKI_LOG = PROJECT_ROOT / "logs/experiments/RUN65_science_nature_wiki.json"

# Se i file sono in logs/ invece che logs/experiments, usa:
# SCI_Q_LOG = PROJECT_ROOT / "logs/RUN300_science_nature.json"
# WIKI_LOG = PROJECT_ROOT / "logs/RUN65_science_nature_wiki.json"

def flatten_experiment(path, run_name):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    for session_idx, session in enumerate(data.get("sessions", [])):
        questions = session.get("questions", [])

        for q_idx, q in enumerate(questions):
            correct = q.get("correct")

            # ignora domande senza giudizio
            if correct is None:
                continue

            options = q.get("options", [])
            options_text = " | ".join(
                f"{o.get('id')}. {o.get('text')}" for o in options
            )

            rows.append({
                "run": run_name,
                "session_idx": session_idx,
                "q_idx_in_game": q_idx,
                "level": q.get("level"),
                "question_id": q.get("question_id"),
                "text": q.get("text"),
                "options": options_text,
                "answer": q.get("answer"),
                "correct": bool(correct),
                "answer_time": q.get("answer_time"),
                "tools_used": ",".join(q.get("tools_used", [])),
            })

    return pd.DataFrame(rows)

sciq = flatten_experiment(SCI_Q_LOG, "sciq")
wiki = flatten_experiment(WIKI_LOG, "wiki")

print("SciQ rows:", len(sciq), "unique questions:", sciq["question_id"].nunique())
print("Wiki rows:", len(wiki), "unique questions:", wiki["question_id"].nunique())

In [ ]:
def aggregate_by_question(df, prefix):
    tmp = df.copy()
    tmp["wrong"] = ~tmp["correct"]

    agg = tmp.groupby("question_id").agg(
        text=("text", "first"),
        options=("options", "first"),
        min_level=("level", "min"),
        max_level=("level", "max"),
        attempts=("question_id", "size"),
        correct_n=("correct", "sum"),
        wrong_n=("wrong", "sum"),
        answers=("answer", lambda x: sorted(set(x.dropna().astype(int).tolist()))),
    ).reset_index()

    agg = agg.rename(columns={
        "attempts": f"{prefix}_attempts",
        "correct_n": f"{prefix}_correct_n",
        "wrong_n": f"{prefix}_wrong_n",
        "answers": f"{prefix}_answers",
        "min_level": f"{prefix}_min_level",
        "max_level": f"{prefix}_max_level",
    })

    return agg

sciq_q = aggregate_by_question(sciq, "sciq")
wiki_q = aggregate_by_question(wiki, "wiki")

overlap = sciq_q.merge(
    wiki_q,
    on="question_id",
    how="outer",
    suffixes=("_sciq", "_wiki"),
)

for col in [
    "sciq_attempts", "sciq_correct_n", "sciq_wrong_n",
    "wiki_attempts", "wiki_correct_n", "wiki_wrong_n",
]:
    overlap[col] = overlap[col].fillna(0).astype(int)

overlap["text"] = overlap["text_sciq"].fillna(overlap["text_wiki"])
overlap["options"] = overlap["options_sciq"].fillna(overlap["options_wiki"])

def classify(row):
    sciq_seen = row["sciq_attempts"] > 0
    wiki_seen = row["wiki_attempts"] > 0

    sciq_has_correct = row["sciq_correct_n"] > 0
    wiki_has_correct = row["wiki_correct_n"] > 0

    if sciq_seen and wiki_seen:
        if not sciq_has_correct and not wiki_has_correct:
            return "both_wrong"
        if not sciq_has_correct and wiki_has_correct:
            return "sciq_wrong_wiki_right"
        if sciq_has_correct and not wiki_has_correct:
            return "wiki_wrong_sciq_right"
        return "both_have_correct"

    if sciq_seen and not wiki_seen:
        return "only_seen_by_sciq"

    if wiki_seen and not sciq_seen:
        return "only_seen_by_wiki"

    return "unknown"

overlap["bucket"] = overlap.apply(classify, axis=1)

cols = [
    "bucket",
    "question_id",
    "text",
    "options",
    "sciq_attempts",
    "sciq_correct_n",
    "sciq_wrong_n",
    "sciq_answers",
    "wiki_attempts",
    "wiki_correct_n",
    "wiki_wrong_n",
    "wiki_answers",
    "sciq_min_level",
    "wiki_min_level",
]

overlap_table = overlap[cols].sort_values(
    ["bucket", "sciq_min_level", "wiki_min_level", "question_id"],
    na_position="last"
)

summary = overlap_table["bucket"].value_counts().rename_axis("bucket").reset_index(name="n")

display(summary)
display(overlap_table.head(100))

OUT = PROJECT_ROOT / "logs/science_nature_sciq_vs_wiki_overlap.csv"
overlap_table.to_csv(OUT, index=False)
print("saved:", OUT)

In [ ]:
display(overlap_table[overlap_table["bucket"] == "both_wrong"])
display(overlap_table[overlap_table["bucket"] == "sciq_wrong_wiki_right"])
display(overlap_table[overlap_table["bucket"] == "wiki_wrong_sciq_right"])